In [1]:
import pandas as pd
import numpy as np
import json
import os

# Exploratory Analysis

In [ ]:
## Load the data with the target columns
# !cut -f 1,8,15,18,21,22,26,67,68,71 /home/camilababo/Documents/DNAquaIMG/BOLD_Public.29-Nov-2024/BOLD_Public.29-Nov-2024.tsv | gzip -c - > bold-selected.tsv.gz

In [31]:
# Load the data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected.tsv"
bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_7321/1221080447.py:3: DtypeWarning: Columns (1,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [4]:
bold.head()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
0,AANIC003-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
1,AANIC002-10,BOLD:AAO2553,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0,COI-5P
2,AANIC058-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
3,AANIC061-10,BOLD:ACE8261,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
4,AANIC062-10,BOLD:AAF6782,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P


In [4]:
bold.tail()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
17862992,ZYGMO209-10,BOLD:AAJ5548,Arthropoda,Zygaenidae,Adscita,Adscita obscura,NaN,AACACTTTACTTTATTTTTGGTATTTGATCAGGAATAGTTGGAACA...,658.0,COI-5P
17862993,ZYGMO290-10,BOLD:AAO1273,Arthropoda,Zygaenidae,Rhagades,Rhagades predotae,NaN,----------------------------------------------...,608.0,COI-5P
17862994,ZYGMO294-10,BOLD:AAO0371,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris minna,NaN,AACACTTTATTTTATTTTTGGAATTTGATCAGGAATAATTGGTACA...,658.0,COI-5P
17862995,ZYGMO295-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P
17862996,ZYGMO296-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P


In [17]:
# Check the total number of entries
bold.shape

(17862997, 10)

#### Check entries on 'Identification Method'

In [15]:
non_null_id = bold[bold['identification_method'] == 'Wikipidia']
non_null_id

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
654690,AZSDS020-18,NaN,Arthropoda,Cerambycidae,NaN,NaN,Wikipidia,NaN,NaN,NaN


In [18]:
# Check the number of entries with COI-5P
count = bold['marker_code'].str.contains('COI-5P', na=False).sum()
int(count)

14790363

In [19]:
# Check the average length of entrie's nucleotide sequence
average_length = bold['nuc'].astype(str).apply(len).mean()
int(average_length)

610

In [13]:
# Check values with unique entries for 'identification method'
unique_idm = bold['identification_method'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/idm.txt",
           unique_idm, fmt='%s')

In [ ]:
# Check values with unique entries for 'marker code'
unique_mc = bold['marker_code'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/mc.txt",
           unique_mc, fmt='%s')

# Pre-processing

#### Filter out entries whose 'marker_code' is not 'COI-5P'

In [3]:
bold = bold[bold['marker_code'] == 'COI-5P']

#### Remove all entries whose sequence is inferior to 500 nucleotides

In [4]:
bold = bold[bold['nuc'].notna() & (bold['nuc_basecount'] > 500.0)]

#### Create column indicating highest taxon level available

In [5]:
bold['highest_tax_level'] = bold[['species', 'genus', 'family', 'phylum']].bfill(axis=1).iloc[:, 0]

#### Filter out entries whose presence of degenerate nucleotides is superior to 1% of the total sequence length

In [6]:
bold['N_count'] = bold['nuc'].str.count('N')

In [7]:
bold['prct_degenerate'] = (bold['N_count'] / bold['nuc_basecount']) * 100

In [8]:
bold = bold[bold['prct_degenerate'] <= 1]

In [10]:
bold.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-init-preprocess.tsv", sep="\t", index=False)

#### Prepare data for taxonomic harmonization against GBIF

In [11]:
# Select highest taxonomic level column to run Tax. Harmonizer
bold_highest_tax = bold[['highest_tax_level']]
bold_highest_tax.rename(columns={'highest_tax_level': 'taxa'}, inplace=True)
bold_highest_tax.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax.tsv", sep="\t", index=False)

# Process Harmonization

In [3]:
# Load the harmonized data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax_harmonized.tsv"
bold_harmonized_taxa = pd.read_csv(path, on_bad_lines="skip", sep="\t")

In [38]:
# Preview the data
bold_harmonized_taxa.head()

,taxa,scientificName,rank,kingdom,phylum,class,order,family,genus
0,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
1,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
2,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
3,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
4,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia


In [28]:
# Check the total number of entries that retrieved no hits '-'
no_hit = bold_harmonized_taxa[bold_harmonized_taxa['scientificName'] == '-']
no_hit_taxa = no_hit['taxa'].to_list()
no_harmonized_entry_percent = len(no_hit_taxa) / len(bold_harmonized_taxa) * 100
f"Number of entries present in the dataset that did not retrieve a harmonized entry: {round(no_harmonized_entry_percent, 1)}%"

'Number of entries present in the dataset that did not retrieve a harmonized entry: 2.3%'

In [29]:
no_hit_taxa_unique = no_hit['taxa'].unique()
f"Number of unique taxa present in the dataset that did not retrieve a harmonized entry: {len(no_hit_taxa_unique)}"

'Number of unique taxa present in the dataset that did not retrieve a harmonized entry: 19458'

In [45]:
# # Write the unique taxa to a file for external checking
# with open("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/no_gbif_hit_taxa.txt", "w") as file:
#     for entry in no_hit_taxa_unique:
#         file.write(entry + "\n")

### Merge pre-processed BOLD with harmonized entries

In [4]:
# Load the initial pre-process data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-init-preprocess.tsv"
bold_init_process = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_4479/1678772608.py:3: DtypeWarning: Columns (1,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_init_process = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [5]:
new_cols = ['processid', 'bin_uri', 'identification_method', 'marker_code', 'nuc', 'nuc_basecount', 'highest_tax_level']
bold_init_process = bold_init_process[new_cols]

In [6]:
# Merge bold_init_process and bold_harmonized_taxa
merged_df = bold_init_process.join(bold_harmonized_taxa)


In [7]:
final_cols = ['processid', 'bin_uri', 'scientificName', 'rank', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'identification_method', 'marker_code', 'nuc', 'nuc_basecount']
merged_df = merged_df[final_cols]

#### Remove entries with missing harmonized entries

In [8]:
merged_cleaned_df = merged_df[merged_df['scientificName'] != '-']

### Process OTLs

##### List OTL's Phylums

In [9]:
harmonized_folder_path = "/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized"

In [10]:
def get_unique_phylum_values(folder_path: str) -> set:
    """
    Recursively reads all .tsv files in a folder, and subfolders, and extracts unique values from
    the 'phylum' column, and returns a set of unique values.

    Parameters:
        folder_path (str): Path to the folder containing .tsv files.

    Returns:
        set: A set of unique values from the 'phylum' column across all files.
    """
    unique_phyla = set()

    for root, _, files in os.walk(folder_path):
        for file_name in files:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(root, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'phylum' in df.columns:
                        # Add unique values from the 'phylum' column to the set
                        unique_phyla.update(df['phylum'].dropna().unique())
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    return unique_phyla

In [11]:
unique_phyla = get_unique_phylum_values(harmonized_folder_path)
unique_phyla.remove('-')

In [12]:
unique_phyla

{'Annelida',
 'Arthropoda',
 'Bryozoa',
 'Charophyta',
 'Chlorophyta',
 'Chordata',
 'Cnidaria',
 'Cyanobacteria',
 'Entoprocta',
 'Mollusca',
 'Nematoda',
 'Nematomorpha',
 'Nemertea',
 'Ochrophyta',
 'Platyhelminthes',
 'Porifera',
 'Rhodophyta',
 'Tardigrada',
 'Tracheophyta'}

In [13]:
# Remove 'chordata'. Only phylum associated with inertebrates (BMI) and unicellular organisms (DIA) are to be considered.
unique_phyla.remove('Chordata')

#### Exploratory analysis on entries where no harmonized entry was retrieved ('-')

In [22]:
def calculate_missing_scientific_names(folder_path) -> pd.DataFrame:
    """
    Calculate the percentage of missing scientific names in .tsv files within a folder and its subfolders.

    Parameters:
        folder_path (str): Path to the folder containing .tsv files.

    Returns:
        pd.DataFrame: A DataFrame containing the file name and the percentage of missing scientific names.
    """
    results = []

    for dirpath, _, filenames in os.walk(folder_path):
        for file_name in filenames:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(dirpath, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'scientificName' in df.columns:
                        total_entries = len(df)
                        missing_entries = (df['scientificName'] == '-').sum()

                        percentage_missing = (missing_entries / total_entries) * 100 if total_entries > 0 else 0

                        file_name = os.path.basename(file_path)

                        results.append({
                            'File Name': file_name,
                            'Percentage Missing': round(percentage_missing, 1)
                        })
                    else:
                        results.append({
                            'File Name': file_name,
                            'Percentage Missing': 'Column not found'
                        })
                except Exception as e:
                    results.append({
                        'File Name': file_name,
                        'Percentage Missing': f'Error: {e}'
                    })

    return pd.DataFrame(results)

In [25]:
stats_on_missing_entries = calculate_missing_scientific_names(harmonized_folder_path)

In [29]:
stats_on_missing_entries.sort_values(by="Percentage Missing", ascending=False)

,File Name,Percentage Missing
7,France_taxonlists_bmi_harmonized.tsv,8.7
1,Austria_taxonlist_bmi_harmonized.tsv,8.6
2,Poland_taxonlists_bmi_harmonized.tsv,4.9
0,Sweden_taxonlist_bmi_harmonized.tsv,3.8
4,Germany_taxonlist_bmi_harmonized.tsv,3.8
10,Czech_taxonlist_diatoms_harmonized.tsv,3.8
5,Portugal_taxonlist_bmi_harmonized.tsv,2.9
6,Finland_taxonlist_bmi_harmonized.tsv,2.7
9,Ireland_taxonlist_diatoms_harmonized.tsv,2.6
11,Finland_taxonlist_diatoms_harmonized.tsv,2.5


#### Remove entries where no harmonized entry was retrieved ('-')

In [30]:
def remove_missing_scientific_names(folder_path) -> None:
    """
    Remove rows with missing scientific names ('-') from .tsv files within a folder and its subfolders.

    Parameters:
        folder_path (str): Path to the folder containing .tsv files

    """
    for dirpath, _, filenames in os.walk(folder_path):
        for file_name in filenames:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(dirpath, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'scientificName' in df.columns:
                        cleaned_df = df[df['scientificName'] != '-']

                        cleaned_df.to_csv(file_path, sep='\t', index=False)
                except Exception as e:
                    print(f"Error processing file {file_name}: {e}")

In [31]:
remove_missing_scientific_names(harmonized_folder_path)

#### Filter BOLD by OTL phylum set

In [14]:
def filter_dataframe_by_set_values(df, phylum_set):
    """
    Filters the BOLD DataFrame to keep only rows where the values in the 'phylum' column
    are in the provided set of allowed phylum values from the OTL.

    Parameters:
    - df (pd.DataFrame): The DataFrame to filter.
    - phylum_set (set): The set of allowed values for the phylum.

    Returns:
    - pd.DataFrame: The filtered DataFrame.
    """
    if 'phylum' in df.columns:
        return df[df['phylum'].isin(phylum_set)]
    else:
        raise ValueError(f"Column 'phylum' not found in the DataFrame.")

In [15]:
merged_cleaned_phyl_filtered_df = filter_dataframe_by_set_values(merged_cleaned_df, unique_phyla)

In [ ]:
# Save file by chunks to avoid memory issues due to large file size
chunk_size = 100000
with open("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-pre-processed.tsv", "w") as f:
    merged_cleaned_phyl_filtered_df.head(0).to_csv(f, sep="\t", index=False)
    for i in range(0, len(merged_cleaned_phyl_filtered_df), chunk_size):
        merged_cleaned_phyl_filtered_df.iloc[i:i+chunk_size].to_csv(f, sep="\t", index=False, header=False)

#### Check values with unique entries for 'identification method'

In [20]:
unique_idm_counts = merged_cleaned_phyl_filtered_df['identification_method'].value_counts()

unique_idm_counts.to_csv(
    "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/final-unique-idm.tsv",
    sep='\t',
    header=['Count'],
    index_label='Identification Method'
)

# Additional Pre-processing

#### Check entries with gaps in sequence

In [3]:
# Load the initial pre-process data
path = "/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-pre-processed.tsv"
bold_pre_processed = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_3913377/2974763359.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_pre_processed = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [5]:
count_with_gap = bold_pre_processed['nuc'].str.contains('-').sum()

f"Number of entries whose sequence contain gaps: {int(count_with_gap)}"

'Number of entries whose sequence contain gaps: 2918763'

In [6]:
total_entries = int(bold_pre_processed.shape[0])
(count_with_gap/total_entries) * 100
f"Percentage of entries whose sequence contain gaps: {round((count_with_gap/total_entries) * 100, 1)}%"

'Percentage of entries whose sequence contain gaps: 22.9%'

In [ ]:
bold_pre_processed['nuc'] = bold_pre_processed['nuc'].str.replace('-', '')

In [6]:
bold_pre_processed.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-pre-processed-gapped-seqs.tsv", sep="\t", index=False)

## Normalize 'identification_method' 

In [8]:
# Load lates pre-processed data package
bold_processed_gaps = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-pre-processed-gapped-seqs.tsv", sep="\t")

/tmp/ipykernel_3913377/142342545.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_processed_gaps = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-pre-processed-gapped-seqs.tsv", sep="\t")


In [11]:
idm_normal = '/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/unique_idm_normalization.tsv'
id_method_normalization = pd.read_csv(idm_normal, sep='\t')

In [12]:
# Create mapping dictionary for identification
method_mapping = id_method_normalization.set_index('Identification Method')['Category'].to_dict()

In [13]:
bold_processed_gaps['category'] = bold_processed_gaps['identification_method'].map(method_mapping).fillna('Unknown')

## Populate 'species' column

In [17]:
bold_processed_gaps["rank"] = bold_processed_gaps["rank"].str.lower()
bold_processed_gaps["rank"] = bold_processed_gaps["rank"].replace({"form": "species", "variety": "species", "subspecies": "species"})
bold_processed_gaps["species"] = np.where(
bold_processed_gaps["rank"] == "species", bold_processed_gaps["scientificName"], np.nan
)
bold_processed_gaps["species"] = bold_processed_gaps["species"].str.split().str[:2].str.join(" ")

In [18]:
bold_processed_gaps = bold_processed_gaps[['processid', 'bin_uri', 'scientificName', 'rank', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'identification_method', 'category', 'marker_code', 'nuc', 'nuc_basecount']]

In [19]:
bold_processed_gaps.to_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-id-species-normalized.tsv", sep="\t", index=False)

## De-replicate (per species)

In [20]:
# Count number of entries for each unique family-genus-species and sequence combination
duplicate_counts = bold_processed_gaps.groupby(["family", "genus", "species", "nuc"]).size().reset_index(name="count")

# Filter only duplicate entries (count > 1)
redundant_entries = duplicate_counts[duplicate_counts["count"] > 1]

print(f"Total redundant entries: {redundant_entries['count'].sum() - len(redundant_entries)}") # extra copies beyond the one we want
print("\nTop 10 groups with redundancies):")
print(redundant_entries.sort_values(by="count", ascending=False).head(10))

Total redundant entries: 1595236

Top 10 groups with redundancies):
                 family         genus                   species  \
674861    Entomobryidae  Lepidocyrtus    Lepidocyrtus paradoxus   
669295    Entomobryidae    Entomobrya        Entomobrya nivalis   
1494645        Phoridae     Megaselia  Megaselia torbjornekremi   
670448    Entomobryidae    Entomobrya        Entomobrya nivalis   
313966     Chironomidae    Limnophyes     Limnophyes asquamatus   
1495628        Phoridae     Megaselia   Megaselia wendyporrasae   
535158        Culicidae         Culex             Culex pipiens   
1226938        Muscidae        Helina           Helina depuncta   
1747315  Sphaeroceridae    Rachispoda    Rachispoda fuscinervis   
669528    Entomobryidae    Entomobrya        Entomobrya nivalis   

                                                       nuc  count  
674861   AACCCTTTATTTAATTTTCGGTGTCTGATCTGCAATAGTTGGAACT...   3629  
669295   TACCTTATATTTAATTTTCGGAGTTTGGGCCGCGATAGTGGGAACT...

In [21]:
# Drop duplicates based on Family-Genus-Species-Nuc while keeping the first one
bmi_derep = bold_processed_gaps.drop_duplicates(subset=["family", "genus", "species", "nuc"], keep="first").reset_index(drop=True)

In [22]:
bmi_derep.to_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-derep.tsv")

In [30]:
bmi_derep.head()

,processid,bin_uri,scientificName,rank,kingdom,phylum,class,order,family,genus,species,identification_method,category,marker_code,nuc,nuc_basecount
0,AANIC003-10,BOLD:AAB9307,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0
1,AANIC002-10,BOLD:AAO2553,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0
2,AANIC058-10,BOLD:AAB9307,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0
3,AANIC061-10,BOLD:ACE8261,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0
4,AANIC062-10,BOLD:AAF6782,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0


## Cluster sequences

In [2]:
bold_derep = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-derep.tsv", sep='\t', index_col=0)

/tmp/ipykernel_3943100/2903759778.py:1: DtypeWarning: Columns (2,12) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_derep = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-derep.tsv", sep='\t', index_col=0)


In [31]:
# Transform database to FASTA for clustering intake (sequence + header with processid)
with open('bold_derep.fasta', "w") as file:
    for index, row in bmi_derep.iterrows():
        file.write(f">{row['processid']}\n{row['nuc']}\n")

In [ ]:
# Command used for clustering
# cd-hit -i /storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold_derep.fasta -o /storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold_clustered.fasta -c 0.99 -M 0 -T 30

In [3]:
# Process clustering file
# Turn clusters into dictionary
# {
#    "cluster1": {
#        "representative-id": "id",
#        "representative-seq": "seq",
#        "clustered-ids": ["id1", "id2"]
#    },
#    "cluster2": {...}
#}

clustering_mapping = {}

with open("bold_clustered.fasta.clstr", 'r', encoding='UTF-8') as file:
    current_cluster = None
    for line in file:
        line = line.strip()

        if line.startswith(">Cluster"):
            current_cluster = f"cluster{line.split()[1]}"
            clustering_mapping[current_cluster] = {"representative-id": "", "representative-seq": "", "clustered-ids": []}
        else:
            parts = line.split('\t')
            seq_info = parts[1]
            seq_id = seq_info.split(", >")[-1].split("...")[0]

            if seq_info.endswith("*"):
                clustering_mapping[current_cluster]["representative-id"] = seq_id
            else:
                clustering_mapping[current_cluster]["clustered-ids"].append(seq_id)

# Remove clusters with only one sequence
clustering_mapping = {k: v for k, v in clustering_mapping.items() if len(v["clustered-ids"]) > 0}

with open("clustering-mapping.json", "w", encoding="utf-8") as f:
    json.dump(clustering_mapping, f, indent=4)

In [ ]:
# Populate representative-seq

with open("clustering-mapping.json", "r", encoding="utf-8") as f:
    clustering_mapping = json.load(f)

seq_id_lookup = dict(zip(bold_derep["processid"], bold_derep["nuc"]))

for cluster in clustering_mapping.values():
    rep_id = cluster["representative-id"]
    if rep_id in seq_id_lookup:
        cluster["representative-id"] = seq_id_lookup[rep_id]

with open("clustering-mapping-updated.json", "w", encoding="utf-8") as f:
    json.dump(clustering_mapping, f, indent=4)

In [ ]:
# Change sequence for clustered ids to the representative seq
id_to_rep_seq = {}

with open("clustering-mapping-updated.json", "r", encoding="utf-8") as f:
    clustering_mapping = json.load(f)

for cluster in clustering_mapping.values():
    rep_seq = cluster["representative-seq"]
    for seq_id in cluster["clustered-ids"]:
        id_to_rep_seq[seq_id] = rep_seq  # map each clustered id to the representative sequence

# replace sequence where 'processid' matches
bold_derep["nuc"] = bold_derep["processid"].map(id_to_rep_seq).fillna(bold_derep["nuc"])

bold_derep.to_csv("bold_clustered.tsv", index=False, sep='\t')

In [5]:
bold_clustered = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold_clustered.csv")

/tmp/ipykernel_3943100/4062324176.py:1: DtypeWarning: Columns (1,11) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_clustered = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold_clustered.csv")


In [12]:
# Redo de-replication
# Drop duplicates based on Family-Genus-Species-Nuc while keeping the first one
bold_clustered_derep = bold_clustered.drop_duplicates(subset=["family", "genus", "species", "nuc"], keep="first").reset_index(drop=True)

In [19]:
reduction = bold_clustered.shape[0] - bold_clustered_derep.shape[0]
print(f"Number of sequences pre-clustering and derep is {bold_clustered.shape[0]}, while the number of sequences after is {bold_clustered_derep.shape[0]}; removing {reduction} duplicate entries.")

Number of sequences pre-clustering and derep is 6849847, while the number of sequences after is 1521118; removing 5328729 duplicate entries.


In [15]:
bold_clustered_derep.to_csv("bold_clustered_derep.tsv", index=False, sep='\t')

# Genetic Biodiversity Analysis between 1M VS 8M

In [1]:
import pandas as pd
bold_derep = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-derep.tsv", sep='\t', index_col=0)

FileNotFoundError: [Errno 2] No such file or directory: '/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold-derep.tsv'

In [ ]:
# Check biodiversity on non-filtered dataset
bold_all_idm = bold_derep[['family', 'genus', 'species']]
bmi_nflit_grouped_df = bold_all_idm.groupby(["family", "genus", "species"]).size().reset_index(name="count")
bmi_nflit_grouped_df.sort_values(by="count", ascending=False)

,family,genus,species,count
58993,Entomobryidae,Entomobrya,Entomobrya nivalis,5799
178854,Tephritidae,Bactrocera,Bactrocera dorsalis,4635
163459,Sciaridae,Scatopsciara,Scatopsciara atomaria,3906
45356,Culicidae,Aedes,Aedes vexans,3105
29310,Chironomidae,Paratanytarsus,Paratanytarsus laccophilus,2982
...,...,...,...,...
133387,Nymphalidae,Algiachroa,Algiachroa woodfordi,1
133403,Nymphalidae,Amauris,Amauris ochlea,1
133405,Nymphalidae,Amauris,Amauris vashti,1
133407,Nymphalidae,Amiga,Amiga sericeella,1


In [2]:
# Check biodiversity on most correct idm
best_idm = ['Morphotaxonomy', 'Integrative taxonomy']
bold_best_idm = bold_derep[bold_derep['category'].isin(best_idm)]
bold_best_idm = bold_best_idm[['family', 'genus', 'species']]
bold_best_idm_grouped_df = bold_best_idm.groupby(["family", "genus", "species"]).size().reset_index(name="count")
bold_best_idm_grouped_df.sort_values(by="count", ascending=False)

NameError: name 'bold_derep' is not defined

In [34]:
# Check which entries are in one but not the other
# In all_idm but not in best_idm
bold_all_idm_nodup = bold_all_idm.drop_duplicates()
bold_best_idm_nodup = bold_best_idm.drop_duplicates()
lost_taxa = bold_all_idm_nodup[~bold_all_idm_nodup.apply(tuple, 1).isin(bold_best_idm_nodup.apply(tuple, 1))]
lost_taxa

,family,genus,species
Unnamed: 0,,,
7,Crambidae,Friedlanderia,Friedlanderia phaeochorda
9,Crambidae,Tawhitia,Tawhitia pentadactylus
10,Crambidae,Corynophora,Corynophora argentifascia
11,Crambidae,Corynophora,Corynophora torrentellus
12,Crambidae,Corynophora,Corynophora lativittalis
...,...,...,...
6849618,Zygaenidae,Jordanita,Jordanita syriaca
6849635,Zygaenidae,Zygaena,Zygaena formosa
6849672,Zygaenidae,Zygaena,Zygaena afghana


In [37]:
# Cretate hierarchy JSON from df
hierarchy = {}

for _, row in lost_taxa.iterrows():
    family = row['family']
    genus = row['genus']
    species = row['species']

    if family not in hierarchy:
        hierarchy[family] = {}

    if genus not in hierarchy[family]:
        hierarchy[family][genus] = []

    hierarchy[family][genus].append(species)

with open("lost_taxa.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy, f, indent=4, ensure_ascii=False)


# Transform to FASTA

In [3]:
final_bmi = pd.read_csv("/storage/archive1/homes/camila.babo/DNAquaIMG/previous_bold_package/bold_clustered_derep.tsv", sep='\t')

In [4]:
final_bmi.head()

,processid,bin_uri,scientificName,rank,kingdom,phylum,class,order,family,genus,species,identification_method,category,marker_code,nuc,nuc_basecount
0,AANIC003-10,BOLD:AAB9307,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0
1,AANIC002-10,BOLD:AAO2553,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0
2,AANIC061-10,BOLD:ACE8261,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0
3,AANIC062-10,BOLD:AAF6782,"Arhodia lasiocamparia Guenée, 1857",species,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,Unknown,COI-5P,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0
4,AANIC073-10,BOLD:AAP3269,"Friedlanderia phaeochorda (Turner, 1911)",species,Animalia,Arthropoda,Insecta,Lepidoptera,Crambidae,Friedlanderia,Friedlanderia phaeochorda,NaN,Unknown,COI-5P,AACTTTATATTTTATTTTTGGAATTTGAGCAGGTATAGTAGGAACA...,658.0


In [7]:
with open("bmi_database.fasta", "w") as fasta_file:
    for _, row in final_bmi.iterrows():
        fasta_file.write(f">{row['processid']}|{row['species']}|{row['scientificName']}|{row['rank']}|{row['kingdom']}|{row['phylum']}|{row['class']}|{row['order']}|{row['family']}|{row['genus']}|{row['species']}\n{row['nuc']}\n")

## Subset based on most accurate 'identification_method'

In [8]:
best_idm = ['Morphotaxonomy', 'Integrative taxonomy']
idm_subset = final_bmi[final_bmi['category'].isin(best_idm)]

In [12]:
with open("bmi_database_idm_subset.fasta", "w") as fasta_file:
    for _, row in idm_subset.iterrows():
        fasta_file.write(f">{row['processid']}|{row['species']}|{row['scientificName']}|{row['rank']}|{row['kingdom']}|{row['phylum']}|{row['class']}|{row['order']}|{row['family']}|{row['genus']}|{row['species']}\n{row['nuc']}\n")